In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [28]:
wine_df = pd.read_csv(r"/home/emad-salib/Wine-Quality-Prediction/data/winequality-red.csv")

In [29]:
wine_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [30]:
wine_df["quality"] = wine_df["quality"].apply(lambda x: 1 if x >= 6 else 0)

In [31]:
wine_df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0


In [32]:
wine_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [33]:
wine_df.drop(["residual sugar", "free sulfur dioxide"], axis=1 , inplace=True)
wine_df.head()

,fixed acidity,volatile acidity,citric acid,chlorides,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,0.076,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,0.098,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,0.092,54.0,0.9970,3.26,0.65,9.8,0
3,11.2,0.28,0.56,0.075,60.0,0.9980,3.16,0.58,9.8,1
4,7.4,0.70,0.00,0.076,34.0,0.9978,3.51,0.56,9.4,0


In [34]:
wine_df.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
chlorides               0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [35]:
wine_df.duplicated().sum()

np.int64(240)

In [36]:
wine_df.drop_duplicates(inplace=True)

In [37]:
wine_df.duplicated().sum()

np.int64(0)

In [38]:
wine_df.describe()

,fixed acidity,volatile acidity,citric acid,chlorides,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000,1359.000000
mean,8.310596,0.529478,0.272333,0.088124,46.825975,0.996709,3.309787,0.658705,10.432315,0.529065
std,1.736990,0.183031,0.195537,0.049377,33.408946,0.001869,0.155036,0.170667,1.082065,0.499338
min,4.600000,0.120000,0.000000,0.012000,6.000000,0.990070,2.740000,0.330000,8.400000,0.000000
25%,7.100000,0.390000,0.090000,0.070000,22.000000,0.995600,3.210000,0.550000,9.500000,0.000000
50%,7.900000,0.520000,0.260000,0.079000,38.000000,0.996700,3.310000,0.620000,10.200000,1.000000
75%,9.200000,0.640000,0.430000,0.091000,63.000000,0.997820,3.400000,0.730000,11.100000,1.000000
max,15.900000,1.580000,1.000000,0.611000,289.000000,1.003690,4.010000,2.000000,14.900000,1.000000


In [39]:
def detect_outliers_iqr(df):
    outlier_indices = []

    for col in df.columns:
        if col == 'quality':
            continue

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower) | (df[col] > upper)].index
        outlier_indices.extend(outliers)

    return set(outlier_indices)

In [40]:
outliers = detect_outliers_iqr(wine_df)

wine_df_clean = wine_df.drop(index=outliers)

print("Original:", wine_df.shape)
print("After removing outliers:", wine_df_clean.shape)

Original: (1359, 10)
After removing outliers: (1119, 10)


In [45]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [47]:
X = wine_df.drop('quality' , axis=1)
y = wine_df['quality']

# Train and split data

In [48]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [49]:
X_train.shape, y_train.shape

((1087, 9), (1087,))

In [50]:
X_test.shape , y_test.shape

((272, 9), (272,))

In [51]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [52]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [53]:
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted)
    recall = recall_score(true, predicted)
    f1 = f1_score(true, predicted)
    return accuracy, precision, recall, f1


In [54]:
models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGBoost": XGBClassifier(),
    "CatBoost": CatBoostClassifier(verbose=False)
}

In [55]:
model_list = []
accuracy_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate Train and Test dataset
    model_train_acc , model_train_prec, model_train_rec, model_train_f1 = evaluate_model(y_train, y_train_pred)

    model_test_acc , model_test_prec, model_test_rec, model_test_f1 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(model_train_acc))
    print("- Precision: {:.4f}".format(model_train_prec))
    print("- Recall: {:.4f}".format(model_train_rec))
    print("- F1 Score: {:.4f}".format(model_train_f1))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Accuracy: {:.4f}".format(model_test_acc))
    print("- Precision: {:.4f}".format(model_test_prec))
    print("- Recall: {:.4f}".format(model_test_rec))
    print("- F1 Score: {:.4f}".format(model_test_f1))

    accuracy_list.append(model_test_acc)
    
    print('='*35)
    print('\n')

Logistic Regression
Model performance for Training set
- Accuracy: 0.7415
- Precision: 0.7654
- Recall: 0.7457
- F1 Score: 0.7554
----------------------------------
Model performance for Test set
- Accuracy: 0.7574
- Precision: 0.7483
- Recall: 0.7810
- F1 Score: 0.7643


KNN
Model performance for Training set
- Accuracy: 0.8086
- Precision: 0.8106
- Recall: 0.8385
- F1 Score: 0.8243
----------------------------------
Model performance for Test set
- Accuracy: 0.7390
- Precision: 0.7292
- Recall: 0.7664
- F1 Score: 0.7473


Decision Tree
Model performance for Training set
- Accuracy: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- F1 Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0.6618
- Precision: 0.6490
- Recall: 0.7153
- F1 Score: 0.6806


Random Forest
Model performance for Training set
- Accuracy: 1.0000
- Precision: 1.0000
- Recall: 1.0000
- F1 Score: 1.0000
----------------------------------
Model performance for Test set
- Accuracy: 0

In [56]:
pd.DataFrame(
    list(zip(model_list , accuracy_list)),
    columns=['Model Name' , 'F1_Score']
).sort_values(by=['F1_Score'] , ascending=False)

,Model Name,F1_Score
6,CatBoost,0.783088
5,XGBoost,0.783088
3,Random Forest,0.775735
4,AdaBoost,0.768382
0,Logistic Regression,0.757353
1,KNN,0.738971
2,Decision Tree,0.661765
